# Comparing Feature Importance Methods

Different XAI methods can give different importance rankings for the same model. Understanding why helps choose the right method for your use case. We compare four approaches on the same model and data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
import shap

# Set random seed for reproducibility
np.random.seed(42)

In [ ]:
# Load Breast Cancer dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train RandomForest classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Print test accuracy
test_accuracy = model.score(X_test, y_test)
print(f"Test Accuracy: {test_accuracy:.4f}")

## Method 1: Impurity-based (MDI)

Built-in to tree models, measures total decrease in impurity from splits on each feature.

In [ ]:
# Extract built-in feature importances (MDI)
mdi_importances = model.feature_importances_
mdi_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': mdi_importances
}).sort_values('Importance', ascending=False)

print("MDI Top 10 Features:")
print(mdi_df.head(10))

## Method 2: Permutation Importance

Shuffles each feature and measures drop in test accuracy.

In [ ]:
# Compute permutation importance
perm_importance = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=42)
perm_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': perm_importance.importances_mean
}).sort_values('Importance', ascending=False)

print("Permutation Importance Top 10 Features:")
print(perm_df.head(10))

## Method 3: SHAP Values

Computes each feature's marginal contribution using Shapley values from game theory.

In [ ]:
# Compute SHAP values
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# For binary classification, take absolute mean SHAP values
# shap_values is a list with 2 arrays (one per class), take the positive class
mean_abs_shap = np.abs(shap_values[1]).mean(axis=0)
shap_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': mean_abs_shap
}).sort_values('Importance', ascending=False)

print("SHAP Top 10 Features:")
print(shap_df.head(10))

## Method 4: Drop-Column Importance

Retrain the model without each feature and measure performance drop (expensive but direct).

In [ ]:
# Drop-column importance (only for top 10 from permutation importance to keep it fast)
top_10_features = perm_df.head(10)['Feature'].tolist()

drop_col_importances = {}
baseline_accuracy = model.score(X_test, y_test)

for feature in top_10_features:
    # Create dataset without this feature
    X_test_without = X_test.drop(columns=[feature])
    X_train_without = X_train.drop(columns=[feature])
    
    # Retrain model
    temp_model = RandomForestClassifier(n_estimators=100, random_state=42)
    temp_model.fit(X_train_without, y_train)
    
    # Measure accuracy drop
    new_accuracy = temp_model.score(X_test_without, y_test)
    importance = baseline_accuracy - new_accuracy
    drop_col_importances[feature] = importance

drop_col_df = pd.DataFrame({
    'Feature': list(drop_col_importances.keys()),
    'Importance': list(drop_col_importances.values())
}).sort_values('Importance', ascending=False)

print("Drop-Column Importance (Top 10 from Permutation):")
print(drop_col_df)

## Side-by-Side Comparison

In [ ]:
# Create 2x2 subplot with top 10 features from each method
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# MDI
mdi_top = mdi_df.head(10).sort_values('Importance')
axes[0, 0].barh(mdi_top['Feature'], mdi_top['Importance'], color='steelblue')
axes[0, 0].set_title('Method 1: Impurity-based (MDI)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Importance Score')

# Permutation Importance
perm_top = perm_df.head(10).sort_values('Importance')
axes[0, 1].barh(perm_top['Feature'], perm_top['Importance'], color='darkorange')
axes[0, 1].set_title('Method 2: Permutation Importance', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Accuracy Drop')

# SHAP
shap_top = shap_df.head(10).sort_values('Importance')
axes[1, 0].barh(shap_top['Feature'], shap_top['Importance'], color='crimson')
axes[1, 0].set_title('Method 3: SHAP Values', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Mean Absolute SHAP')

# Drop-Column
drop_col_top = drop_col_df.sort_values('Importance')
axes[1, 1].barh(drop_col_top['Feature'], drop_col_top['Importance'], color='seagreen')
axes[1, 1].set_title('Method 4: Drop-Column Importance', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Accuracy Drop')

plt.tight_layout()
plt.show()

print("\nComparison plot created successfully.")

In [ ]:
# Create ranking comparison DataFrame for top 10 features from permutation importance
top_10_features_from_perm = perm_df.head(10)['Feature'].tolist()

# Create ranking dataframe (rank 1 = most important)
comparison_df = pd.DataFrame()
comparison_df['Feature'] = top_10_features_from_perm

# Get ranks for each method
mdi_ranks = pd.Series(range(1, len(mdi_df) + 1), index=mdi_df['Feature'])
perm_ranks = pd.Series(range(1, len(perm_df) + 1), index=perm_df['Feature'])
shap_ranks = pd.Series(range(1, len(shap_df) + 1), index=shap_df['Feature'])
drop_col_ranks = pd.Series(range(1, len(drop_col_df) + 1), index=drop_col_df['Feature'])

comparison_df['MDI Rank'] = comparison_df['Feature'].map(mdi_ranks)
comparison_df['Permutation Rank'] = comparison_df['Feature'].map(perm_ranks)
comparison_df['SHAP Rank'] = comparison_df['Feature'].map(shap_ranks)
comparison_df['Drop-Column Rank'] = comparison_df['Feature'].map(drop_col_ranks)

# Calculate average rank
comparison_df['Avg Rank'] = comparison_df[['MDI Rank', 'Permutation Rank', 'SHAP Rank', 'Drop-Column Rank']].mean(axis=1)
comparison_df = comparison_df.sort_values('Avg Rank')

print("\nRanking Comparison (Top 10 from Permutation Importance):")
print("Lower rank number = more important")
print(comparison_df.to_string(index=False))

## Key Takeaways

- **MDI is biased toward high-cardinality and correlated features**: Tree-based importance measures favor features used for early splits, which may not reflect true predictive power.

- **Permutation importance can underestimate importance of correlated features**: When features are highly correlated, removing one may not hurt performance because another correlated feature can compensate.

- **SHAP provides the most theoretically grounded attributions**: Based on Shapley values from game theory, SHAP offers a principled framework for feature contributions with desirable properties like local accuracy and consistency.

- **Drop-column is the most direct measure but prohibitively expensive for large models**: Retraining the model for each feature is computationally costly but gives the most realistic view of each feature's impact.

- **When methods agree on important features, you can be more confident; disagreements warrant investigation**: Use multiple methods to validate findings and investigate cases where methods disagree significantly.